# Langfuse v3 — Dataset & Run Explorer / Exporter (REST edition)

Explore datasets, runs, run items (with linked traces + reasoning), then export to **Excel** and **JSON**.

This version talks to the Langfuse **Public REST API** directly via `httpx`, bypassing the
SDK's `Langfuse()` client entirely. That sidesteps the self-signed / TLS-proxy SSL problems
in the OTel exporter — every call here goes through one `httpx.Client(verify=False, trust_env=False)`.

**Workflow:** configure → list datasets → pick one → explore (items only *or* runs+traces+reasoning) → export.

## 1. Connection (REST client)

In [ ]:
import httpx

LANGFUSE_HOST       = "https://your-langfuse.internal"   # no trailing slash
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

_client = httpx.Client(
    base_url=LANGFUSE_HOST,
    auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
    verify=False,
    trust_env=False,
    timeout=httpx.Timeout(30.0, read=120.0),
)

def lf_get(path, **params):
    """GET a Langfuse public API endpoint, e.g. lf_get('/api/public/v2/datasets', page=1)."""
    r = _client.get(path, params={k: v for k, v in params.items() if v is not None})
    r.raise_for_status()
    return r.json()

h = _client.get("/api/public/health"); h.raise_for_status()
print("Connected OK ->", LANGFUSE_HOST, "|", h.json())

## 2. Imports & helpers

In [ ]:
import json, datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd

OUT_DIR = Path("langfuse_exports"); OUT_DIR.mkdir(exist_ok=True)

def _ts(): return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _flatten(value, max_len=None):
    """Make a value Excel-safe: dict/list -> compact JSON string, optional truncation."""
    if value is None: return None
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    value = str(value)
    if max_len and len(value) > max_len:
        value = value[:max_len] + " ...[truncated]"
    return value

def paginate(path, data_key="data", **params):
    """Walk all pages of a paginated public-API list endpoint."""
    page, out = 1, []
    while True:
        js = lf_get(path, page=page, limit=100, **params)
        data = js.get(data_key, []) or []
        if not data: break
        out.extend(data)
        meta = js.get("meta") or {}
        total_pages = meta.get("totalPages")
        if total_pages is not None and page >= total_pages: break
        if total_pages is None and len(data) < 100: break
        page += 1
    return out

print("Helpers ready. Exports ->", OUT_DIR.resolve())

## 3. List all datasets
Browse what's on the project, then copy a name into the next cell.

In [ ]:
datasets = paginate("/api/public/v2/datasets")

df_datasets = pd.DataFrame([{
    "name": d.get("name"),
    "id": d.get("id"),
    "description": d.get("description"),
    "metadata": _flatten(d.get("metadata")),
    "createdAt": d.get("createdAt"),
    "updatedAt": d.get("updatedAt"),
} for d in datasets])

print(f"{len(df_datasets)} dataset(s) found\n")
df_datasets

## 4. Pick a dataset
Set `DATASET_NAME`. Everything below operates on this dataset.

In [ ]:
DATASET_NAME = "your-dataset-name"   # <-- edit me

dataset = lf_get(f"/api/public/v2/datasets/{quote(DATASET_NAME, safe='')}")
print("Dataset:", dataset.get("name"))
print("ID:     ", dataset.get("id"))
print("Desc:   ", dataset.get("description"))

## 5. Mode A — Dataset items only

Inputs / expected outputs / metadata. No runs, no traces.

> Note: we filter by `datasetName` (not `datasetId`) — a known public-API bug makes the
> `datasetId` filter silently return items from *all* datasets.

In [ ]:
items = paginate("/api/public/dataset-items", datasetName=DATASET_NAME)

df_items = pd.DataFrame([{
    "item_id": it.get("id"),
    "status": it.get("status"),
    "input": _flatten(it.get("input")),
    "expected_output": _flatten(it.get("expectedOutput")),
    "metadata": _flatten(it.get("metadata")),
    "source_trace_id": it.get("sourceTraceId"),
    "source_observation_id": it.get("sourceObservationId"),
    "createdAt": it.get("createdAt"),
} for it in items])

print(f"{len(df_items)} item(s)")
df_items.head(20)

## 6. List runs for this dataset
In v3 a "dataset run" == an "experiment".

In [ ]:
runs = paginate(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs")

df_runs = pd.DataFrame([{
    "run_name": r.get("name"),
    "run_id": r.get("id"),
    "description": r.get("description"),
    "metadata": _flatten(r.get("metadata")),
    "createdAt": r.get("createdAt"),
    "updatedAt": r.get("updatedAt"),
} for r in runs])

print(f"{len(df_runs)} run(s) for '{DATASET_NAME}'")
df_runs

## 7. Mode B — Runs + outputs + trace reasoning

Pick which runs to pull. For each run item we fetch the linked **trace** and its
**observations** (generations) so you get the model's actual output and intermediate reasoning.

- `SELECTED_RUNS = []` → all runs
- `SELECTED_RUNS = ["run-a", "run-b"]` → only those
- `INCLUDE_OBSERVATIONS = True` → pull every observation per trace (reasoning chain)

In [ ]:
SELECTED_RUNS = []              # [] = all runs, or e.g. ["baseline-v1", "gpt4o-run"]
INCLUDE_OBSERVATIONS = True     # fetch per-observation reasoning (slower, richer)
TEXT_LIMIT = 8000               # truncate long fields for Excel safety (None = no limit)

target = df_runs["run_name"].tolist() if not SELECTED_RUNS else SELECTED_RUNS
print(f"Pulling {len(target)} run(s)...")

# caches to avoid refetching shared traces/items
_trace_cache, _obs_cache, _item_cache = {}, {}, {}

def get_run_with_items(run_name):
    """GET /datasets/{name}/runs/{runName} returns the run plus its run items."""
    return lf_get(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs/{quote(run_name, safe='')}")

def get_trace(trace_id):
    if not trace_id: return None
    if trace_id not in _trace_cache:
        try: _trace_cache[trace_id] = lf_get(f"/api/public/traces/{trace_id}")
        except Exception as e:
            _trace_cache[trace_id] = None; print(f"  ! trace {trace_id}: {e}")
    return _trace_cache[trace_id]

def get_dataset_item(item_id):
    if not item_id: return None
    if item_id not in _item_cache:
        try: _item_cache[item_id] = lf_get(f"/api/public/dataset-items/{item_id}")
        except Exception: _item_cache[item_id] = None
    return _item_cache[item_id]

def get_observations(trace_id):
    if not trace_id: return []
    if trace_id not in _obs_cache:
        try:
            _obs_cache[trace_id] = paginate("/api/public/observations", traceId=trace_id)
        except Exception as e:
            _obs_cache[trace_id] = []; print(f"  ! obs {trace_id}: {e}")
    return _obs_cache[trace_id]

flat_rows = []   # one row per run item (Excel)
nested = []      # full structured tree (JSON)

for run_name in target:
    run_full = get_run_with_items(run_name)
    run_items = run_full.get("datasetRunItems") or run_full.get("data") or []
    print(f"  run '{run_name}': {len(run_items)} item(s)")
    run_node = {"run": {k: v for k, v in run_full.items() if k not in ("datasetRunItems", "data")},
                "items": []}

    for it in run_items:
        trace_id = it.get("traceId")
        item_id  = it.get("datasetItemId")

        di = get_dataset_item(item_id)
        expected = di.get("expectedOutput") if di else None

        trace = get_trace(trace_id)
        observations = get_observations(trace_id) if INCLUDE_OBSERVATIONS else []

        reasoning_parts = []
        for o in observations:
            out = o.get("output")
            if out is not None:
                reasoning_parts.append(f"[{o.get('type','')}:{o.get('name','')}] {_flatten(out, 2000)}")
        reasoning = "\n".join(reasoning_parts) if reasoning_parts else None

        flat_rows.append({
            "run_name": run_name,
            "run_item_id": it.get("id"),
            "dataset_item_id": item_id,
            "trace_id": trace_id,
            "observation_id": it.get("observationId"),
            "input": _flatten(trace.get("input") if trace else None, TEXT_LIMIT),
            "actual_output": _flatten(trace.get("output") if trace else None, TEXT_LIMIT),
            "expected_output": _flatten(expected, TEXT_LIMIT),
            "reasoning_chain": _flatten(reasoning, TEXT_LIMIT),
            "n_observations": len(observations),
            "trace_name": trace.get("name") if trace else None,
            "latency_ms": trace.get("latency") if trace else None,
            "total_cost": trace.get("totalCost") if trace else None,
        })

        run_node["items"].append({
            "run_item": it,
            "dataset_item": di,
            "trace": trace,
            "observations": observations,
        })

    nested.append(run_node)

df_run_items = pd.DataFrame(flat_rows)
print(f"\nTotal run items pulled: {len(df_run_items)}")
df_run_items.head(20)

## 8. Quick inspection
Look at one full record before exporting.

In [ ]:
if not df_run_items.empty:
    row = df_run_items.iloc[0]
    print("RUN     :", row["run_name"])
    print("TRACE   :", row["trace_id"])
    print("\nINPUT\n", str(row["input"])[:1500])
    print("\nEXPECTED\n", str(row["expected_output"])[:1500])
    print("\nACTUAL\n", str(row["actual_output"])[:1500])
    print("\nREASONING CHAIN\n", str(row["reasoning_chain"])[:3000])
else:
    print("No run items - run Mode B (section 7) first.")

## 9. Export to Excel + JSON

Toggle what to write. Each export is timestamped under `langfuse_exports/`.
- **Excel**: one workbook, sheets = datasets / items / runs / run_items
- **JSON**: flat run-items + full nested tree (traces + observations, untruncated)

In [ ]:
EXPORT_DATASETS    = True
EXPORT_ITEMS       = True
EXPORT_RUNS        = True
EXPORT_RUN_ITEMS   = True
EXPORT_NESTED_JSON = True

stamp = _ts()
safe = quote(DATASET_NAME, safe="")

# ---- Excel ----
xlsx_path = OUT_DIR / f"{safe}_{stamp}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xl:
    wrote = False
    if EXPORT_DATASETS and 'df_datasets' in dir() and not df_datasets.empty:
        df_datasets.to_excel(xl, sheet_name="datasets", index=False); wrote = True
    if EXPORT_ITEMS and 'df_items' in dir() and not df_items.empty:
        df_items.to_excel(xl, sheet_name="items", index=False); wrote = True
    if EXPORT_RUNS and 'df_runs' in dir() and not df_runs.empty:
        df_runs.to_excel(xl, sheet_name="runs", index=False); wrote = True
    if EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
        df_run_items.to_excel(xl, sheet_name="run_items", index=False); wrote = True
    if not wrote:
        pd.DataFrame({"note": ["nothing selected/empty"]}).to_excel(xl, sheet_name="empty", index=False)
print("Excel  ->", xlsx_path)

# ---- JSON ----
payload = {"exported_at": stamp, "host": LANGFUSE_HOST, "dataset_name": DATASET_NAME}
if EXPORT_ITEMS and 'df_items' in dir():
    payload["items"] = df_items.to_dict(orient="records")
if EXPORT_RUNS and 'df_runs' in dir():
    payload["runs"] = df_runs.to_dict(orient="records")
if EXPORT_RUN_ITEMS and 'df_run_items' in dir():
    payload["run_items_flat"] = df_run_items.to_dict(orient="records")

json_path = OUT_DIR / f"{safe}_{stamp}.json"
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
print("JSON   ->", json_path)

if EXPORT_NESTED_JSON and 'nested' in dir() and nested:
    nested_path = OUT_DIR / f"{safe}_{stamp}_nested.json"
    nested_path.write_text(json.dumps(nested, ensure_ascii=False, indent=2, default=str))
    print("Nested ->", nested_path)

print("\nDone.")

---
### Notes (REST edition)

- **No SDK / no OTel** → none of the self-signed SSL issues apply. All traffic uses the one
  verified `httpx` client from cell 1.
- **`dataset-items` filter**: use `datasetName=`, *not* `datasetId=` — the id filter is
  silently ignored by the public API and returns items across all datasets.
- **Reasoning** comes from per-observation `output`s on each trace. If your app doesn't emit
  intermediate generations you'll only get trace-level `input`/`output`.
- **Pagination** keys: public API returns `{"data": [...], "meta": {"totalPages": N, ...}}`.
- Self-hosted ingestion lags ~15-30s; very recent runs may not appear immediately.
- If `/v2/datasets` 404s on an older platform build, swap to the v1 route `/api/public/datasets`.